In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.head())

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

In [2]:
col = ["PassengerId", "Ticket", "Cabin"]
df = df.drop(columns = col)

In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    str    
 3   Sex       891 non-null    str    
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Fare      891 non-null    float64
 8   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(3)
memory usage: 62.8 KB
None


In [4]:
df['FamilySize'] = df["SibSp"] + df['Parch'] + 1

In [5]:
print(df[["SibSp", "Parch", "FamilySize"]].head())

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1


In [7]:
df['Title'] = df['Name'].str.extract(r" ([A-Za-z]+)\.", expand = False)
print(df['Title'].value_counts())

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


In [9]:
title_mapping = {
    "Mlle": "Miss", "Ms": "Miss",
    "Mme": "Mrs",
    "Dr": "Officer", "Major": "Officer","Rev": "Officer", "Col": "Officer", "Capt": "Officer",
    "Don": "Royalty", "Sir": "Royalty", "Lady": "Royalty", "Countess": "Royalty", "Jonkheer": "Royalty"
}

df["Title"] = df['Title'].replace(title_mapping)
print(df["Title"].value_counts())


Title
Mr         517
Miss       185
Mrs        126
Master      40
Officer     18
Royalty      5
Name: count, dtype: int64


In [10]:
df = df.drop(columns= ["Name"])

In [11]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    str    
 3   Age         714 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Fare        891 non-null    float64
 7   Embarked    889 non-null    str    
 8   FamilySize  891 non-null    int64  
 9   Title       891 non-null    str    
dtypes: float64(2), int64(5), str(3)
memory usage: 69.7 KB
None


In [ ]:
df['Pclass'] = df["Pclass"].astype(str)

In [ ]:
df['Sex'] = df['Sex'].map({"male": 0, "female": 1})

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
numeric_features = ["Age", "SibSp", 'Parch', 'Fare', 'FamilySize', 'Sex']
categorical_features =['Pclass', 'Embarked', "Title"]

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop = 'first'))
])

In [ ]:
age_median = X_train["Age"].median()
X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

In [ ]:
emb = X_train["Embarked"].mode()[0]
X_train["Embarked"] = X_train["Embarked"].fillna(emb)
X_test["Embarked"] = X_test["Embarked"].fillna(emb)

In [ ]:
viz_df = pd.concat([X_train, y_train],axis = 1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = viz_df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot = True, cmap = 'coolwarm')
plt.title("Coorelation Heatmap")

In [ ]:
sns.histplot(data = viz_df, x = 'Age', hue = "Sex", bins = 20, kde = True)
plt.show()

In [ ]:
X_train["Sex"] = X_train["Sex"].astype(str)
X_test['Sex'] = X_test['Sex'].astype(str)
X_train['Embarked'] = X_train['Embarked'].astype(str)
X_test['Embarked'] = X_test['Embarked'].astype(str)

In [ ]:
numcols = X_train.select_dtypes(include = 'number').columns
print(numcols)
strcols = X_train.select_dtypes(include = 'str').columns
print(strcols)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numcols] = scaler.fit_transform(X_train_scaled[numcols])
X_test_scaled[numcols] = scaler.transform(X_test_scaled[numcols])

In [ ]:
X_train_scaled = pd.get_dummies(X_train_scaled, columns= strcols, drop_first = True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns=strcols, drop_first=True)

X_train_scaled, X_test_scaled = X_train_scaled.align(X_test_scaled, join = 'left', axis = 1, fill_value=0)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight= "balanced" ),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced")
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    print(f"{name}: Test: {test_acc} | Train: {train_acc}")

In [ ]:
from sklearn.metrics import classification_report

for name, model in models.items():
    print(name)
    print(classification_report(y_test, model.predict(X_test_scaled)))

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv = 5)
    print(f"{name}: {scores.mean()} +/- {scores.std()}")

In [ ]:
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2,5]   
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    rf_param_grid,
    cv = 5,
    n_jobs = -1,
    scoring = 'accuracy'
)

grid.fit(X_train_scaled, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_model =  grid.best_estimator_
print(best_model.score(X_test_scaled, y_test)) 

In [ ]:
importances = pd.Series(best_model.feature_importances_, index = X_train_scaled.columns)
print(importances.sort_values())

In [ ]:
import joblib

joblib.dump(best_model, "Titanic.pkl")

In [ ]:
loaded_model = joblib.load("Titanic.pkl")
print(loaded_model.score(X_test_scaled, y_test))

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = loaded_model.predict(X_test_scaled)
cm =  confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot = True, fmt = 'd', xticklabels=["Died", "Survived"], yticklabels=["Died", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()